In [1]:
import torch
import sentencepiece
from torch.distributions import Categorical
from torchmetrics.text import BLEUScore,SacreBLEUScore

Decoding 

In [2]:
import importlib
import train
importlib.reload(train)
from train import DEVICE, config, load_best_model

In [3]:
model, _ = load_best_model()



num_parameters: 32,986,112 - num_non_embedding_params: 29,402,112
2026-11-05 16:30:00.029953: loss_fn loaded
2026-11-05 16:30:00.029992: loading best pre saved
best model path found.
2026-11-05 16:30:00.463304: Starting from pre saved with: start_epoch: 24, best_val_loss: 2.9024, last_epoch_vloss: 2.90, non_improve_cnt: 0.00, best_epoch: 24


In [4]:
sp_bpe = sentencepiece.SentencePieceProcessor()
sp_bpe.load(f"tokenizers/{config["tokenizer_source"]}/bpe.model")
sacreblue_metric = SacreBLEUScore(lowercase=True)
blue_metric = BLEUScore()

In [6]:
english_orig_translation = "Last year I went skydiving with my friends. There were cows in the sky."
german_text = "Letztes Jahr bin ich mit meinen Freunden Fallschirm gesprungen. Da waren Kühe am Himmel."


german_list_enc = sp_bpe.encode_as_ids(german_text, add_bos=True, add_eos=True)
german_encoding,pmask = model.encode(torch.tensor([german_list_enc]).to(DEVICE))

english_encoding = [sp_bpe.bos_id()]

i = 0
model.eval()
with torch.no_grad():
    while i < 128:
        logits = model.decode(german_encoding,torch.tensor([english_encoding]).to(DEVICE),pmask) # B,Sy,VS
        # dist = Categorical(logits=logits[0,-1,:])

        # new_enc = dist.sample().item()
        new_enc = logits[0,-1,:].argmax().item()
        english_encoding.append(new_enc)
        if new_enc == sp_bpe.eos_id():
            print(f"debug: new encoding: '{sp_bpe.decode_ids(english_encoding)}'")
            print("debug: reached EOS")
            break 
    
        if i % 5 == 0:
            print(f"debug: new encoding: '{sp_bpe.decode_ids(english_encoding)}'")
        i+=1
sacre_score = sacreblue_metric([sp_bpe.decode_ids(english_encoding)], [[english_orig_translation]])
score = blue_metric([sp_bpe.decode_ids(english_encoding)], [[english_orig_translation]])
print(f"SacreBLEU Score: {sacre_score}")
print(f"BLEU Score: {score}")

debug: new encoding: 'L'
debug: new encoding: 'Last year I was with'
debug: new encoding: 'Last year I was with my friends. There were'
debug: new encoding: 'Last year I was with my friends. There were at the coast.'
debug: new encoding: 'Last year I was with my friends. There were at the coast.'
debug: reached EOS
SacreBLEU Score: 0.40592148900032043
BLEU Score: 0.3696446418762207


In [ ]:
# 1.7s -> 0.2s

Batch calculations and comparing with original paper

In [ ]:
from train import load_data, lr_update_fn
import torch

In [ ]:
config["mega_batch_size"]

In [ ]:
train_loader, val_loader = load_data()

In [ ]:
total_x = 0
total_y = 0
x_len=0
y_len=0

pad_tokens = 0
for X,Y in train_loader: # B,Sy
    cur_x = torch.sum(X != config["pad_token"])
    cur_y = torch.sum(Y != config["pad_token"])
    pad_tokens += X.shape[0] * (X.shape[1] + Y.shape[1]) - cur_x - cur_y
    total_x+=cur_x
    total_y+=cur_y

    x_len+=X.shape[1]
    y_len+=Y.shape[1]

In [ ]:
print(f"total_x: {total_x.item():,}")
print(f"total_y: {total_y.item():,}")
print(f"total_pad_tokens: {pad_tokens.item():,}")

print("per step: ")
print(f"total_x: {total_x / len(train_loader):,.2f}")
print(f"total_y: {total_y / len(train_loader):,.2f}")
print(f"pad_tokens: {pad_tokens / len(train_loader) / 2:,.2f}")
print(f"X_len: {x_len / len(train_loader):.2f}")
print(f"Y_len: {y_len / len(train_loader):.2f}")

In [ ]:
batches_per_epoch_me = 3221 # from logs/output_logs_overnight_Sep_5_2.txt
num_epochs_me = 15
sent_len_me = (x_len + y_len)/(2*len(train_loader))
sent_len_them = sent_len_me #??
print(f"tokens seen total: {100_000 * 25_000:,} - {(total_x) * num_epochs_me:,}")
print(f"tokens/batch:      {25_000:,} - {(total_x + total_y)/len(train_loader):,.0f}")
print(f"tokens/epoch:      {4_500_000 * sent_len_them:,.0f} - {total_x + total_y:,}") # assuming they have the same sentence length as us
print(f"batches/epoch:     {(4_500_000 * sent_len_them / 25_000):,.0f} - {batches_per_epoch_me:,}") # batches_t * tokens/epoch / tokens_t
print(f"epochs:            {100_000 / ((4_500_000 * sent_len_them / 25_000)):.1f} - {num_epochs_me}")
print(f"batches:           {100_000:,} - {batches_per_epoch_me * num_epochs_me:,.2f}")

took back: 

- higher tokens/batch than paper -> less gradient noise. Hence better. 
- batches/epoch is half -> Noam LR is still warming up at the start
- total batches is third -> Noam LR is not fully calmed down yet at the end -> lower warmup_steps. 
- 

Calculating BLEU

In [26]:
import importlib
import train
importlib.reload(train)
from train import calculate_bleu, load_best_model, load_data

In [27]:
train_loader, val_loader, train_token_count = load_data(sp_bpe)
print(f"{train_token_count:,}")
del train_loader

2026-10-08 00:54:30.714932: encoding train started
2026-10-08 00:54:30.716263 cache found
2026-10-08 00:54:30.889942: encoding val started
2026-10-08 00:54:30.890144 cache found
2026-10-08 00:54:30.891134: # train size: #Tokens: X: 5,802,911 Y: 5,662,684 #Sentences: 204,226, num_dropped: 0
2026-10-08 00:54:30.891147: # val size: #Tokens: X: 27,929 Y: 26,864 #Sentences: 883, num_dropped: 0
2026-10-08 00:54:30.891155: batches/epoch: 6,383. LR peak: 0.0008068715304598784
5,802,911


/Users/mohammadaminsedaghat/Desktop/attention/train.py:192: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  self.X = torch.from_numpy(X)


In [18]:
model, _, history = load_best_model()



num_parameters: 63,045,632 - num_non_embedding_params: 44,101,632
2026-10-03 18:57:56.725196: loss_fn loaded
2026-10-03 18:57:56.725209: loading best pre saved
best model path not found. Loading from: ./test/overnight_Oct_2/model_0_keyboard_intrrupt.pt
2026-10-03 18:57:56.858673: Starting from pre saved with: start_epoch: 0, best_val_loss: inf, last_epoch_vloss: inf, non_improve_cnt: 0.00


In [ ]:
gen = calculate_bleu(val_loader,model,sp_bpe,return_translations=False) 
# epoch 10: 30.61

2026-10-08 00:54:34.415783 started calculating bleu
2026-10-08 00:55:11.766667 finished calculating bleu - avg bleu: 30.61 - total_time: 37.35081505775452 - avg_eos_idx: 27.43
